In [1]:
import sys
import os
import subprocess
import shutil

# Check if we are running in Google Colab (remote Linux VM)
if 'google.colab' in sys.modules:
    print("Detected Google Colab environment. Setting up via GitHub...")
    
    # 1. Install required packages using PyG pre-compiled wheels to avoid long build times
    import torch
    torch_version = torch.__version__.split('+')[0]
    # Format cuda version, e.g. '12.1' -> 'cu121'
    if torch.version.cuda:
        cuda_version = f"cu{torch.version.cuda.replace('.', '')}"
    else:
        cuda_version = "cpu"
        
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html"
    
    print(f"Installing PyG dependencies from {wheel_url}...")
    subprocess.run(["pip", "install", "-q", "torch_geometric", "prody"], check=True)
    subprocess.run(["pip", "install", "-q", "torch-cluster", "-f", wheel_url], check=True)

    # 2. Clone/update main repository
    repo_url = "https://github.com/WSobo/Struct2Seq-GCN.git"
    target_dir = "/content/Struct2Seq-GCN"

    if not os.path.exists(target_dir):
        os.chdir('/content')
        print("Cloning repository...")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        os.chdir(target_dir)
        print("Pulling latest changes...")
        subprocess.run(["git", "pull"], check=True)

    # 3. Ensure LigandMPNN dependency exists (fallback for gitlink/submodule issues)
    ligand_dir = os.path.join(target_dir, "LigandMPNN")
    ligand_data_utils = os.path.join(ligand_dir, "data_utils.py")
    if not os.path.exists(ligand_data_utils):
        print("Fetching LigandMPNN dependency...")
        if os.path.exists(ligand_dir):
            shutil.rmtree(ligand_dir)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/dauparas/LigandMPNN.git", "LigandMPNN"],
            cwd=target_dir,
            check=True,
        )

    # 4. Enter the project directory
    os.chdir(target_dir)
    print("Colab setup complete. Current Dir:", os.getcwd())

Detected Google Colab environment. Setting up via GitHub...
Installing PyG dependencies from https://data.pyg.org/whl/torch-2.10.0+cu128.html...
Pulling latest changes...
Colab setup complete. Current Dir: /content/Struct2Seq-GCN


In [2]:
#@title Which Branch?

!git checkout model-experiments

Already on 'model-experiments'
Your branch is up to date with 'origin/model-experiments'.


# Struct2Seq-GCN: Pipeline Testing
This notebook runs the basic sanity checks for the inference and training pipelines of the Struct2Seq-GCN model.

In [8]:
import os
# Ensure we operate from the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/Struct2Seq-GCN


## Step 1: Test Inference Pipeline (Sanity Check)
Before training, make sure the model can successfully take a 3D structure, convert it to a graph, run it through the GCN, and output sequence predictions.

In [9]:
import os

# Set PYTHONPATH environment variable so python subprocesses can find project modules
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{os.getcwd()}"

# Use an actual PDB file as input for parse_PDB
!python scripts/run.py --pdb LigandMPNN/inputs/1BC8.pdb

Parsing LigandMPNN/inputs/1BC8.pdb...
Predicted Sequence Length: 93
Done!


## Step 2: Test Training Loop
Confirm the data loader can successfully read the LigandMPNN `train.json` splits and start optimizing the weights for a couple of epochs.

In [10]:
import os
import json
import glob

# Build a tiny local split from available input PDBs for a stable smoke test
pdb_files = sorted(glob.glob("LigandMPNN/inputs/*.pdb"))
ids = [os.path.splitext(os.path.basename(p))[0] for p in pdb_files]
if len(ids) < 2:
    raise RuntimeError("Need at least 2 PDB files in LigandMPNN/inputs for train/valid smoke test.")

os.makedirs("training", exist_ok=True)
with open("training/train_smoke.json", "w") as f:
    json.dump([ids[0]], f)
with open("training/valid_smoke.json", "w") as f:
    json.dump([ids[1]], f)

!python scripts/train.py \
    --json_train training/train_smoke.json \
    --json_valid training/valid_smoke.json \
    --pdb_dir LigandMPNN/inputs/ \
    --epochs 2 \
    --batch_size 1

Using device: cuda
Initializing datasets... (This will process raw PDBs into PyG graphs)
Starting training...
Epoch 001 | Train Loss: 3.0889 - Acc: 0.1290 | Val Loss: 3.1938 - Acc: 0.0473
  -> Saved new best model!
Epoch 002 | Train Loss: 2.8067 - Acc: 0.1935 | Val Loss: 3.1770 - Acc: 0.0465
  -> Saved new best model!


In [ ]:
import os

# 1. Clear the PyTorch Geometric processed cache.
# This forces the dataset loader to rebuild the graphs from the raw PDBs using the latest graph_builder.py logic.
!rm -rf training/train_data/processed
!rm -rf training/valid_data/processed

print("Cleared PyG dataset cache!")

# 2. Restart the Colab Kernel (optional, but guarantees a heavily-refreshed environment).
# Uncomment the two lines below if you are experiencing weird CUDA/Python caching issues:
# print("Restarting kernel...")
# os.kill(os.getpid(), 9)

## Step 3: Scale up Training Loop test
Try 50 epochs

In [11]:
!python scripts/train.py \
    --json_train LigandMPNN/training/train.json \
    --json_valid LigandMPNN/training/valid.json \
    --pdb_dir LigandMPNN/inputs/ \
    --epochs 50 \
    --batch_size 16

Using device: cuda
Initializing datasets... (This will process raw PDBs into PyG graphs)
Processing...
Done!
Processing...
Done!
Starting training...
Traceback (most recent call last):
  File "/content/Struct2Seq-GCN/scripts/train.py", line 126, in <module>
    main()
  File "/content/Struct2Seq-GCN/scripts/train.py", line 113, in main
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Struct2Seq-GCN/scripts/train.py", line 19, in train_epoch
    for batch in loader:
                 ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 801, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^